# 🦀 Day 2: Lifetime Elision Rules

Today: When Rust **infers** lifetimes automatically!

---

## 📋 The Three Elision Rules

The compiler applies these rules to figure out lifetimes:

1. **Each reference parameter gets its own lifetime**
   - `fn foo(x: &str, y: &str)` → `fn foo<'a, 'b>(x: &'a str, y: &'b str)`

2. **If there's exactly one input lifetime, it's assigned to all output lifetimes**
   - `fn foo(x: &str) -> &str` → `fn foo<'a>(x: &'a str) -> &'a str`

3. **If `&self` or `&mut self` is a parameter, its lifetime is assigned to all outputs**
   - `fn foo(&self, x: &str) -> &str` → lifetime of `&self` used for output

In [ ]:
// Rule 2: One input reference → output gets same lifetime
// No annotation needed!
fn first_char(s: &str) -> &str {
    &s[..1]
}

fn trim_and_return(s: &str) -> &str {
    s.trim()
}

fn to_bytes(s: &str) -> &[u8] {
    s.as_bytes()
}

println!("{}", first_char("hello"));
println!("{}", trim_and_return("  hello  "));
println!("{:?}", to_bytes("hi"));

In [ ]:
// Rule 3: Methods with &self → output borrows from self
struct TextData {
    content: String,
    author: String,
}

impl TextData {
    // No lifetime needed — Rule 3 applies
    fn get_content(&self) -> &str {
        &self.content
    }

    fn get_author(&self) -> &str {
        &self.author
    }

    fn summary(&self) -> String {  // Returns owned String — no lifetime needed
        format!("{} by {}", &self.content[..20.min(self.content.len())], self.author)
    }

    // Even with extra params, &self lifetime wins for output
    fn get_content_or(&self, _default: &str) -> &str {
        if self.content.is_empty() { _default } else { &self.content }
        // ⚠️ Actually this needs explicit lifetimes if _default could be returned!
    }
}

let data = TextData {
    content: "Rust is amazing".into(),
    author: "Ferris".into(),
};
println!("Content: {}", data.get_content());
println!("Author: {}", data.get_author());
println!("Summary: {}", data.summary());

## 🚫 When Elision Fails

In [ ]:
// Multiple input references + returning a reference → Must annotate!

// This WON'T compile without annotations:
// fn pick(a: &str, b: &str) -> &str { a }

// Must specify which input the output borrows from:
fn pick<'a>(a: &'a str, _b: &str) -> &'a str {
    a  // only borrows from 'a'
}

fn max_str<'a>(a: &'a str, b: &'a str) -> &'a str {
    if a > b { a } else { b }  // could borrow from either
}

println!("{}", pick("hello", "world"));
println!("{}", max_str("apple", "banana"));

## 🧪 Elision Quiz

In [ ]:
// For each function, determine if lifetime annotations are needed:

// 1. fn len(s: &str) -> usize
//    → NO: returns usize (not a reference)
fn len(s: &str) -> usize { s.len() }

// 2. fn first_line(s: &str) -> &str
//    → NO: Rule 2 (one input → output gets same lifetime)
fn first_line(s: &str) -> &str {
    s.lines().next().unwrap_or("")
}

// 3. fn longer(a: &str, b: &str) -> &str
//    → YES: Rule 1 gives 'a and 'b, but which for output?
fn longer<'a>(a: &'a str, b: &'a str) -> &'a str {
    if a.len() >= b.len() { a } else { b }
}

// 4. fn parse(input: &str) -> Result<i32, &str>
//    → NO: Rule 2 (one input → output Err gets same lifetime)
fn parse(input: &str) -> Result<i32, &str> {
    input.parse().map_err(|_| input)
}

println!("len: {}", len("hello"));
println!("first_line: {}", first_line("line1\nline2"));
println!("longer: {}", longer("hi", "hello"));
println!("parse: {:?}", parse("42"));
println!("parse: {:?}", parse("abc"));

## 🏋️ Exercises

In [ ]:
// Exercise 1: For each function signature, state whether
// lifetime annotations are needed and why:
// a) fn split_at(s: &str, pos: usize) -> (&str, &str)
// b) fn get_ref(v: &Vec<i32>, i: usize) -> &i32
// c) fn combine(a: &str, b: &str) -> String
// d) fn pick_one(a: &str, b: &str, first: bool) -> &str

// Write your answers as comments, then implement each:

// YOUR CODE HERE


## 🎯 Key Takeaways

1. **Rule 1**: Each reference param gets its own lifetime
2. **Rule 2**: One input lifetime → applied to all outputs
3. **Rule 3**: `&self`/`&mut self` lifetime → applied to outputs
4. If rules don't resolve all output lifetimes → you must annotate
5. Returning owned types (`String`, `Vec`) **never** needs lifetime annotations

---
**Tomorrow:** Lifetimes in structs! 🏗️